In [4]:
from pathlib import Path
import sys

import cv2
import numpy as np
import torch
import onnxruntime as ort

# Make project imports work from testing/
ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT / "src"))

from models.classifier import DefectClassifier

# ---- Paths ----
CKPT_PATH = ROOT / "checkpoints" / "classifier-epoch=16-val_auroc=0.9226.ckpt"
ONNX_PATH = ROOT / "checkpoints" / "classifier-epoch=16-val_auroc=0.9226.onnx"
IMAGE_PATH = ROOT / "data" / "processed" / "patches" / "bottle" / "test" / "broken_small" / "bottle_test_broken_small_000_patch_0_388.png"

# ---- Load model from checkpoint ----
checkpoint = torch.load(CKPT_PATH, map_location="cpu")
cfg = checkpoint["hyper_parameters"]
model = DefectClassifier.load_from_checkpoint(str(CKPT_PATH), cfg=cfg, map_location="cpu")
model.eval()

# ---- Export to ONNX ----
dummy = torch.randn(1, 3, cfg["data"]["image_size"], cfg["data"]["image_size"], dtype=torch.float32)
torch.onnx.export(
    model,
    dummy,
    str(ONNX_PATH),
    export_params=True,
    opset_version=18,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
)
print(f"Exported ONNX model to: {ONNX_PATH}")

/tmp/ipykernel_119814/4164832634.py:28: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0324 19:34:31.343000 119814 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `DefectClassifier([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `DefectClassifier([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/home/mahdi/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).
Failed to convert the model to the target version 17 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/home/mahdi/Desktop/Projects/NullDefect/.venv/lib/python3.12/site-packages/onnxscript/version_converter/__init__.py", line 127, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/mahdi/Desktop/Projects/NullDefect/.venv/lib/python3.12/site-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/home/mahdi/Desktop/Projects/NullDefect/.venv/lib/python3.12/site-packages/onnxscript/version_converter/__init__.py", line 122, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^

Applied 1 of general pattern rewrite rules.
Exported ONNX model to: /home/mahdi/Desktop/Projects/NullDefect/checkpoints/classifier-epoch=16-val_auroc=0.9226.onnx
Image: /home/mahdi/Desktop/Projects/NullDefect/data/processed/patches/bottle/test/broken_small/bottle_test_broken_small_000_patch_0_388.png
Predicted class: defective
Confidence: 0.5189
Probabilities: normal=0.4811, defective=0.5189


In [8]:
# ---- Preprocess image ----
def preprocess_image(image_path: Path, image_size: int) -> np.ndarray:
    img = cv2.imread(str(image_path))
    if img is None:
        raise FileNotFoundError(f"Image not found: {image_path}")

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (image_size, image_size), interpolation=cv2.INTER_LINEAR)
    img = img.astype(np.float32) / 255.0

    # ImageNet normalization (same as training config)
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    img = (img - mean) / std

    # HWC -> CHW and add batch dimension
    img = np.transpose(img, (2, 0, 1))[None, ...]
    return img.astype(np.float32)

In [11]:
# ---- ONNX inference ----
def infer_onnx(onnx_path: Path, image_path: Path, image_size: int):
    x = preprocess_image(image_path, image_size)

    session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    input_name = session.get_inputs()[0].name
    logits = session.run(None, {input_name: x})[0]

    probs = torch.softmax(torch.from_numpy(logits), dim=1).numpy()[0]
    pred_idx = int(np.argmax(probs))

    classes = {0: "normal", 1: "defective"}
    print(f"Image: {image_path}")
    print(f"Predicted class: {classes.get(pred_idx, pred_idx)}")
    print(f"Confidence: {probs[pred_idx]:.4f}")
    print(f"Probabilities: normal={probs[0]:.4f}, defective={probs[1]:.4f}")




In [30]:
# ---- Run inference on specific image path ----
IMAGE_PATH = ROOT / "data" / "processed" / "patches" / "bottle" / "test" / "broken_large" / "bottle_test_broken_large_004_patch_0_0.png"

# Optional: run preprocessing first (sanity check tensor shape)
x = preprocess_image(IMAGE_PATH, cfg["data"]["image_size"])
print("Preprocessed tensor shape:", x.shape)

# Run ONNX inference
infer_onnx(ONNX_PATH, IMAGE_PATH, cfg["data"]["image_size"])


Preprocessed tensor shape: (1, 3, 512, 512)
Image: /home/mahdi/Desktop/Projects/NullDefect/data/processed/patches/bottle/test/broken_large/bottle_test_broken_large_004_patch_0_0.png
Predicted class: defective
Confidence: 0.9992
Probabilities: normal=0.0008, defective=0.9992


In [24]:
# ---- Run inference on specific image path ----
IMAGE_PATH = ROOT / "data" / "processed" / "patches" / "bottle" / "test" / "good" / "bottle_test_good_001_patch_388_0.png"

# Optional: run preprocessing first (sanity check tensor shape)
x = preprocess_image(IMAGE_PATH, cfg["data"]["image_size"])
print("Preprocessed tensor shape:", x.shape)

# Run ONNX inference
infer_onnx(ONNX_PATH, IMAGE_PATH, cfg["data"]["image_size"])

Preprocessed tensor shape: (1, 3, 512, 512)
Image: /home/mahdi/Desktop/Projects/NullDefect/data/processed/patches/bottle/test/good/bottle_test_good_001_patch_388_0.png
Predicted class: normal
Confidence: 0.8101
Probabilities: normal=0.8101, defective=0.1899


In [25]:
# ---- Run inference on specific image path ----
IMAGE_PATH = ROOT / "data" / "processed" / "patches" / "toothbrush" / "test" / "defective" / "toothbrush_test_defective_000_patch_0_448.png"

# Optional: run preprocessing first (sanity check tensor shape)
x = preprocess_image(IMAGE_PATH, cfg["data"]["image_size"])
print("Preprocessed tensor shape:", x.shape)

# Run ONNX inference
infer_onnx(ONNX_PATH, IMAGE_PATH, cfg["data"]["image_size"])

Preprocessed tensor shape: (1, 3, 512, 512)
Image: /home/mahdi/Desktop/Projects/NullDefect/data/processed/patches/toothbrush/test/defective/toothbrush_test_defective_000_patch_0_448.png
Predicted class: defective
Confidence: 0.9859
Probabilities: normal=0.0141, defective=0.9859


In [26]:
# ---- Run inference on specific image path ----
IMAGE_PATH = ROOT / "data" / "processed" / "patches" / "bottle" / "test" / "broken_small" / "bottle_test_broken_small_000_patch_0_388.png"

# Optional: run preprocessing first (sanity check tensor shape)
x = preprocess_image(IMAGE_PATH, cfg["data"]["image_size"])
print("Preprocessed tensor shape:", x.shape)

# Run ONNX inference
infer_onnx(ONNX_PATH, IMAGE_PATH, cfg["data"]["image_size"])

Preprocessed tensor shape: (1, 3, 512, 512)
Image: /home/mahdi/Desktop/Projects/NullDefect/data/processed/patches/bottle/test/broken_small/bottle_test_broken_small_000_patch_0_388.png
Predicted class: defective
Confidence: 0.5189
Probabilities: normal=0.4811, defective=0.5189


In [28]:
# ---- Run inference on specific image path ----
IMAGE_PATH = ROOT / "data" / "processed" / "patches" / "transistor" / "test" / "good" / "transistor_test_good_000_patch_448_0.png"

# Optional: run preprocessing first (sanity check tensor shape)
x = preprocess_image(IMAGE_PATH, cfg["data"]["image_size"])
print("Preprocessed tensor shape:", x.shape)

# Run ONNX inference
infer_onnx(ONNX_PATH, IMAGE_PATH, cfg["data"]["image_size"])

Preprocessed tensor shape: (1, 3, 512, 512)
Image: /home/mahdi/Desktop/Projects/NullDefect/data/processed/patches/transistor/test/good/transistor_test_good_000_patch_448_0.png
Predicted class: defective
Confidence: 0.5955
Probabilities: normal=0.4045, defective=0.5955


In [6]:
import os
print(os.getcwd())

/home/mahdi/Desktop/Projects/NullDefect/testing


In [8]:
import sys
from pathlib import Path

# Set project root correctly
PROJECT_ROOT = Path("/home/mahdi/Desktop/Projects/NullDefect")
SRC_DIR = PROJECT_ROOT / "src"

# Add src folder to sys.path (so Python sees "steps" as a top-level module)
if str(SRC_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(SRC_DIR.resolve()))

# Test import
try:
    from steps.evaluation.compute_metrics import segmentation_metrics_step
    print("✅ Import successful!")
except ModuleNotFoundError as e:
    print("❌ Import failed:", e)

# Now run segmentation metrics
seg_metrics = segmentation_metrics_step(
    checkpoint_path=str(PROJECT_ROOT / "checkpoints/segmenter-epoch=39-val_dice=0.8553.ckpt"),
    metadata_csv=str(PROJECT_ROOT / "data/processed/metadata/dataset_metadata.csv"),
    device="cpu",
)

print("Segmentation metrics computed:", seg_metrics)

❌ Import failed: No module named 'steps'


NameError: name 'segmentation_metrics_step' is not defined

In [ ]:
import sys
from pathlib import Path

# Path to your NullDefect project root
PROJECT_ROOT = Path("/home/mahdi/Desktop/Projects/NullDefect")
SRC_DIR = PROJECT_ROOT / "src"

# Add src folder to sys.path
sys.path.insert(0, str(SRC_DIR.resolve()))
print("sys.path updated:", sys.path[:3])  # just to check

# Now import
from steps.evaluation.compute_metrics import segmentation_metrics_step
print("✅ Import succeeded")

sys.path updated: ['/home/mahdi/Desktop/Projects/NullDefect/src', '/home/mahdi/Desktop/Projects/NullDefect/src', '/home/Desktop/Projects/NullDefect/src']


ModuleNotFoundError: No module named 'steps'

In [10]:
from steps.evaluation.compute_metrics import segmentation_metrics_step

seg_metrics = segmentation_metrics_step(
    checkpoint_path="/home/mahdi/Desktop/Projects/NullDefect/checkpoints/segmenter-epoch=39-val_dice=0.8553.ckpt",
    metadata_csv="/home/mahdi/Desktop/Projects/NullDefect/data/processed/metadata/dataset_metadata.csv",
    device="cpu",
)

print("Segmentation metrics computed:", seg_metrics)

ModuleNotFoundError: No module named 'steps'

In [2]:
# ---- Export segmenter last.ckpt to ONNX ----
from pathlib import Path
import sys

import torch

cwd = Path.cwd().resolve()
ROOT = cwd if (cwd / "src").exists() else cwd.parent
if not (ROOT / "src").exists():
    raise FileNotFoundError(f"Could not locate project root from {cwd}")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from models.segmentation import DefectSegmenter

SEG_CKPT_PATH = ROOT / "checkpoints" / "last.ckpt"
SEG_ONNX_PATH = ROOT / "checkpoints" / "segmenter-last.onnx"

checkpoint = torch.load(SEG_CKPT_PATH, map_location="cpu")
cfg = checkpoint["hyper_parameters"]
model = DefectSegmenter.load_from_checkpoint(str(SEG_CKPT_PATH), cfg=cfg, map_location="cpu")
model.eval()

image_size = int(cfg["data"]["image_size"])
dummy = torch.randn(1, 3, image_size, image_size, dtype=torch.float32)

with torch.no_grad():
    sample_out = model(dummy)

if isinstance(sample_out, dict):
    output_names = list(sample_out.keys())
elif isinstance(sample_out, (tuple, list)):
    output_names = [f"output_{i}" for i in range(len(sample_out))]
else:
    output_names = ["logits"]

torch.onnx.export(
    model,
    dummy,
    str(SEG_ONNX_PATH),
    export_params=True,
    opset_version=18,
    do_constant_folding=True,
    input_names=["input"],
    output_names=output_names,
    dynamic_axes={"input": {0: "batch"}, **{name: {0: "batch"} for name in output_names}},
)

print(f"Exported ONNX model to: {SEG_ONNX_PATH}")


/home/mahdi/Desktop/Projects/NullDefect/.venv/lib/python3.12/site-packages/lightning_fabric/__init__.py:40: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
/home/mahdi/Desktop/Projects/NullDefect/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_180354/406237537.py:37: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `DefectSegmenter([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `DefectSegmenter([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/home/mahdi/Desktop/Projects/NullDefect/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 NVIDIA GeForce MX330 which is of cuda capability 6.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/home/mahdi/Desktop/Projects/NullDefect/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/home/mahdi/Desktop/Projects/NullDefect/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:435: UserWarning: 
NVIDIA GeForce MX330 with CUDA capability sm_61 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the NVIDIA GeForce MX330 GPU with PyTorch, please check the instructi

[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 33 of general pattern rewrite rules.
Exported ONNX model to: /home/mahdi/Desktop/Projects/NullDefect/checkpoints/segmenter-last.onnx
